In [1]:
import os


In [2]:
%pwd

'c:\\Users\\babar\\OneDrive\\Desktop\\MLops project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\babar\\OneDrive\\Desktop\\MLops project'

In [5]:

# Import the dataclass decorator to automatically generate methods
# such as __init__() and __repr__() for the class.
from dataclasses import dataclass

# Import Path to represent filesystem paths in a platform-independent way.
from pathlib import Path


# Convert this class into a dataclass for storing configuration data.
@dataclass
class DataIngestion_Config:

    # Store the root directory for the data ingestion process.
    root_dir: Path

    # Store the URL from which the dataset will be downloaded.
    source_url: str

    # Store the local filesystem path where the downloaded dataset will be saved.
    local_data_file: Path

    # Store the directory path where the downloaded dataset will be extracted.
    unzip_dir: Path


In [ ]:
	
from src.Mlops_project_quality_predictor.constants import *
from src.Mlops_project_quality_predictor.utils.common import read_yaml,create_directories


In [7]:

# Define a class responsible for managing project configurations.
class ConfigurationManager:

    # Initialize the configuration manager with paths to configuration files.
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,  # Set the default path to the main configuration YAML file.
        params_filepath=PARAMS_FILE_PATH,  # Set the default path to the parameters YAML file.
        schema_filepath=SCHEMA_FILE_PATH,  # Set the default path to the schema YAML file.
    ):
        # Read the main YAML file and store its contents as a ConfigBox object.
        self.config = read_yaml(config_filepath)

        # Read the parameters YAML file and store its contents.
        self.params = read_yaml(params_filepath)

        # Read the schema YAML file and store its contents.
        self.schema = read_yaml(schema_filepath)

        # Create the main artifacts directory defined in the configuration file.
        create_directories([self.config.artifacts_root])

    # Define a method that returns the data ingestion configuration object.
    def get_data_ingestion_config(self) -> DataIngestion_Config:

        # Access the data ingestion section of the main configuration.
        config = self.config.data_ingestion

        # Create the root directory required for the data ingestion pipeline.
        create_directories([config.root_dir])

        # Create a DataIngestion_Config object using values from the YAML file.
        data_ingestion_config = DataIngestion_Config(

            # Set the root directory for data ingestion.
            root_dir=config.root_dir,

            # Set the URL from which the dataset will be downloaded.
            source_url=config.source_url,

            # Set the local path where the downloaded dataset will be stored.
            local_data_file=config.local_data_file,

            # Set the directory where the downloaded dataset will be extracted.
            unzip_dir=config.unzip_dir,
        )

        # Return the completed data ingestion configuration object.
        return data_ingestion_config


In [8]:
import os
import urllib.request as request

from src.Mlops_project_quality_predictor.utils import logger
import zipfile


In [9]:

# Import os for filesystem operations.
import os

# Import zipfile to extract ZIP archives.
import zipfile

# Import urlretrieve to download files from a URL.
from urllib.request import urlretrieve

# Import Path to handle filesystem paths.
from pathlib import Path


# Create a class to manage dataset downloading and extraction.
class DataIngestion:

    # Initialize the class with the data ingestion configuration.
    def __init__(self, config: DataIngestion_Config):
        # Store the configuration object for use in other methods.
        self.config = config

    # Download the dataset ZIP file if it does not already exist.
    def download_file(self):

        # Convert the configured file path to a Path object.
        local_data_file = Path(self.config.local_data_file)

        # Create the parent directory if it does not exist.
        local_data_file.parent.mkdir(parents=True, exist_ok=True)

        # Check whether the dataset file already exists.
        if not local_data_file.exists():

            # Download the file from the configured source URL.
            filename, headers = urlretrieve(
                url=self.config.source_url,
                filename=str(local_data_file)
            )

            # Log the downloaded filename and response headers.
            logger.info(
                f"{filename} downloaded with the following info:\n{headers}"
            )

        else:
            # Log that downloading is unnecessary.
            logger.info("File already exists.")

    # Extract the downloaded ZIP file into the configured directory.
    def extract_zip_file(self):

        # Convert the extraction directory to a Path object.
        unzip_path = Path(self.config.unzip_dir)

        # Create the extraction directory if it does not exist.
        unzip_path.mkdir(parents=True, exist_ok=True)

        # Open the downloaded ZIP archive in read mode.
        with zipfile.ZipFile(
            self.config.local_data_file, "r"
        ) as zip_ref:

            # Extract all archive contents into the target directory.
            zip_ref.extractall(unzip_path)

        # Log successful extraction.
        logger.info(f"Files extracted successfully to: {unzip_path}")


In [19]:

try:
    # Initialize the configuration manager.
    config = ConfigurationManager()

    # Get the data ingestion configuration.
    data_ingestion_config = config.get_data_ingestion_config()

    # Create the DataIngestion object.
    data_ingestion = DataIngestion(data_ingestion_config)

    # Download the dataset.
    data_ingestion.download_file()

    # Extract the ZIP file.
    data_ingestion.extract_zip_file()

except Exception:
    # Re-raise the original error with its traceback.
    raise


[2026-10-10 11:09:22,639:INFO:common:YAML file loaded successfully: C:\Users\babar\OneDrive\Desktop\MLops project\config\config.yaml]
[2026-10-10 11:09:22,645:INFO:common:YAML file loaded successfully: C:\Users\babar\OneDrive\Desktop\MLops project\params.yaml]
[2026-10-10 11:09:22,652:INFO:common:YAML file loaded successfully: C:\Users\babar\OneDrive\Desktop\MLops project\schema.yaml]
[2026-10-10 11:09:22,655:INFO:common:created directory at: artifacts]
[2026-10-10 11:09:22,657:INFO:common:created directory at: artifacts/data_ingestion]
[2026-10-10 11:09:23,104:INFO:540446256:artifacts\data_ingestion\data.zip downloaded with the following info:
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny